# Image Compression - Optimal Quality

**Goal**: Compress images to target size with highest possible quality.

**Strategy**: 
- 🎯 Tries PNG lossless first
- Binary search finds optimal JPEG quality (50-99)
- Converges in ~5 iterations (fast!)

**How to use**:
1. Run all cells (Ctrl+Shift+Enter)
2. Set path and target size in main cell
3. Done! One line: `compress_image(path, target_mb)`

**Output**: `compressed_images/{name}_compressed.jpg`

## Setup and Imports

In [1]:
import os
import sys
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import shutil
import zipfile
from datetime import datetime
import warnings

# Increase PIL's decompression bomb limit for large scientific images
# Default is ~89 million pixels, but scientific images can be much larger
Image.MAX_IMAGE_PIXELS = 500_000_000  # 500 million pixels (adjust if needed)

# Suppress decompression bomb warnings since we're handling it
warnings.filterwarnings("ignore", category=Image.DecompressionBombWarning)

# Create output directory if it doesn't exist
output_dir = Path("compressed_images")
output_dir.mkdir(exist_ok=True)

print("✅ Setup complete!")
print(f"   Output directory: {output_dir.absolute()}")
print(f"   Max image size: {Image.MAX_IMAGE_PIXELS:,} pixels")

✅ Setup complete!
   Output directory: e:\mjosa_complete\gref4hsi\mjosa_code\notebooks\compressed_images
   Max image size: 500,000,000 pixels


## Compression Functions

In [2]:
def get_file_size_mb(file_path):
    """Get file size in megabytes."""
    size_bytes = os.path.getsize(file_path)
    return size_bytes / (1024 * 1024)


def check_image_info(image_path):
    """
    Check image dimensions and file size before compression.
    Useful for very large images.
    """
    img = Image.open(image_path)
    width, height = img.size
    total_pixels = width * height
    file_size_mb = get_file_size_mb(image_path)

    print(f"📏 Image info:")
    print(f"   Dimensions: {width} x {height} pixels")
    print(f"   Total pixels: {total_pixels:,}")
    print(f"   File size: {file_size_mb:.2f} MB")
    print(f"   Format: {img.format}")
    print(f"   Mode: {img.mode}")

    if total_pixels > 100_000_000:
        print(f"\n⚠️  Large image detected!")
        print(f"   This may take a while to compress...")

    return {
        "width": width,
        "height": height,
        "total_pixels": total_pixels,
        "file_size_mb": file_size_mb,
        "format": img.format,
        "mode": img.mode,
    }


def compress_image(
    input_path,
    target_size_mb,
    output_dir="compressed_images",
    print_debug=False,
    max_dimension=None,
):
    """
    Compress image to target size with optimal quality.

    Strategy:
    1. Check if dimensions exceed limit (e.g., Overleaf's 16384 pt limit)
    2. Resize if needed to fit dimension constraints
    3. Try PNG lossless first
    4. Binary search JPEG quality (50-99) to find highest quality under target
    5. Stop when converged or quality too low (< 50)

    Args:
        input_path: Path to input image
        target_size_mb: Target file size in MB
        output_dir: Output directory (default: "compressed_images")
        print_debug: If True, print detailed debug info during compression
        max_dimension: Maximum width or height in pixels (e.g., 16384 for Overleaf)

    Returns:
        Path to compressed image
    """
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    original_size_mb = get_file_size_mb(input_path)

    if print_debug:
        print(f"\n{'='*70}")
        print(f"🔍 DEBUG MODE - Compression Process")
        print(f"{'='*70}")
        print(f"Input: {input_path.name}")
        print(f"Original size: {original_size_mb:.2f} MB")
        print(f"Target size: {target_size_mb:.2f} MB")
        if max_dimension:
            print(f"Max dimension limit: {max_dimension} px")
        print(f"{'='*70}\n")

    # Load and prepare image
    img = Image.open(input_path)
    original_dims = img.size

    # Check if image exceeds dimension limits (e.g., Overleaf)
    if print_debug and max_dimension:
        print(f"📐 Checking dimension limits...")
        print(f"   Current dimensions: {img.size[0]} x {img.size[1]} px")
        print(f"   Max dimension limit: {max_dimension} px")
        max_current_dim = max(img.size[0], img.size[1])
        if max_current_dim > max_dimension:
            print(f"   ❌ EXCEEDS LIMIT by {max_current_dim - max_dimension} px")
        else:
            print(
                f"   ✅ Fits within limit ({max_dimension - max_current_dim} px to spare)"
            )
        print()

    if max_dimension and (img.size[0] > max_dimension or img.size[1] > max_dimension):
        if print_debug:
            print(f"📐 Step 0: Resizing to fit dimension limit...")

        # Calculate scale factor to fit within max_dimension
        scale_factor = max_dimension / max(img.size[0], img.size[1])
        new_width = int(img.size[0] * scale_factor)
        new_height = int(img.size[1] * scale_factor)

        if print_debug:
            print(
                f"   Resizing: {img.size[0]} x {img.size[1]} → {new_width} x {new_height} px"
            )
            print(f"   Scale factor: {scale_factor:.4f}\n")

        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
        original_dims = img.size  # Update dims after resize

    # Convert to RGB if needed (for JPEG)
    if img.mode in ("RGBA", "LA", "P"):
        background = Image.new("RGB", img.size, (255, 255, 255))
        if img.mode == "P":
            img = img.convert("RGBA")
        background.paste(
            img, mask=img.split()[-1] if img.mode in ("RGBA", "LA") else None
        )
        img = background

    # Step 1: Try PNG lossless
    if print_debug:
        print("📦 Step 1: Trying PNG lossless compression...")

    png_path = output_dir / f"{input_path.stem}_temp.png"
    img.save(png_path, "PNG", compress_level=9, optimize=True)
    png_size = get_file_size_mb(png_path)

    if print_debug:
        print(f"   PNG size: {png_size:.2f} MB")
        if png_size <= target_size_mb:
            print(f"   ✅ PNG fits! (≤ {target_size_mb} MB)")
        else:
            print(f"   ❌ PNG too large (> {target_size_mb} MB)")
            print(f"   → Moving to JPEG compression...\n")

    if png_size <= target_size_mb:
        # PNG works! Use it
        final_path = output_dir / f"{input_path.stem}_compressed.png"
        shutil.move(str(png_path), str(final_path))
        print(f"Original: {original_size_mb:.1f} MB")
        print(f"Compressed: {png_size:.1f} MB (PNG lossless)")
        print(f"Saved to: {final_path}")
        return str(final_path)

    # PNG too large, remove it and try JPEG
    png_path.unlink()

    # Step 2: Binary search for optimal JPEG quality
    if print_debug:
        print("🔍 Step 2: Binary search for optimal JPEG quality")
        print(f"   Search range: quality {50}-{99}")
        print(f"   Converge when gap < 2\n")

    min_q = 50  # Quality cutoff
    max_q = 99  # Start at highest quality
    best_quality = None
    best_size = None
    best_path = None
    temp_files = []
    iteration = 0

    while max_q - min_q > 1:  # Converge when gap < 2
        iteration += 1
        mid_q = (min_q + max_q) // 2
        temp_path = output_dir / f"{input_path.stem}_q{mid_q}.jpg"

        img.save(temp_path, "JPEG", quality=mid_q, optimize=True)
        size_mb = get_file_size_mb(temp_path)
        temp_files.append(temp_path)

        if print_debug:
            status = "✅ FITS" if size_mb <= target_size_mb else "❌ TOO LARGE"
            print(
                f"   Iteration {iteration}: Quality {mid_q} → {size_mb:.2f} MB {status}"
            )

        if size_mb <= target_size_mb:
            # This quality works, try higher
            best_quality = mid_q
            best_size = size_mb
            best_path = temp_path
            min_q = mid_q
            if print_debug:
                print(
                    f"      → File fits! Trying higher quality (range now {min_q}-{max_q})"
                )
        else:
            # Too large, go lower
            max_q = mid_q
            if print_debug:
                print(
                    f"      → Too large! Trying lower quality (range now {min_q}-{max_q})"
                )

    if print_debug:
        print(f"\n   🎯 Converged after {iteration} iterations!")
        print(f"   Quality range: {min_q}-{max_q} (gap = {max_q - min_q})\n")

    # Check if we found a solution
    if best_quality is None or best_quality < 50:
        # Need to resize the image
        if print_debug:
            print(f"⚠️  Even quality 50 is too large. Trying to resize image...\n")

        # Try quality 50 to see how much we need to resize
        temp_path = output_dir / f"{input_path.stem}_q50.jpg"
        img.save(temp_path, "JPEG", quality=50, optimize=True)
        size_at_q50 = get_file_size_mb(temp_path)
        temp_path.unlink()

        if print_debug:
            print(f"   Size at quality 50: {size_at_q50:.2f} MB")
            print(f"   Target size: {target_size_mb:.2f} MB")

        # Calculate resize factor (with safety margin)
        size_ratio = target_size_mb / size_at_q50
        scale_factor = (size_ratio * 0.9) ** 0.5  # 0.9 = safety margin

        new_width = int(original_dims[0] * scale_factor)
        new_height = int(original_dims[1] * scale_factor)

        if print_debug:
            print(f"   Original dimensions: {original_dims[0]} x {original_dims[1]}")
            print(f"   New dimensions: {new_width} x {new_height}")
            print(f"   Scale factor: {scale_factor:.3f}\n")
            print(f"📐 Step 4: Resizing and compressing...")

        # Resize image
        resized_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

        # Save with quality 75 (good quality for resized images)
        final_path = output_dir / f"{input_path.stem}_compressed.jpg"
        resized_img.save(final_path, "JPEG", quality=75, optimize=True)
        final_size = get_file_size_mb(final_path)

        # Clean up temp files
        for f in temp_files:
            if f.exists():
                f.unlink()

        if print_debug:
            print(f"{'='*70}")
            print(f"✅ COMPRESSION COMPLETE (with resize)")
            print(f"{'='*70}")
        print(
            f"Original: {original_size_mb:.1f} MB ({original_dims[0]} x {original_dims[1]})"
        )
        print(
            f"Compressed: {final_size:.1f} MB ({new_width} x {new_height}) at quality 75"
        )
        print(f"Saved to: {final_path}")
        if print_debug:
            print(f"{'='*70}\n")
        return str(final_path)
    # Success! Move best file to final location
    if print_debug:
        print(f"📦 Step 3: Saving final compressed image...")
        print(f"   Best quality found: {best_quality}")
        print(f"   Final size: {best_size:.2f} MB")
        print(f"   Cleaning up {len(temp_files) - 1} temp files...\n")

    final_path = output_dir / f"{input_path.stem}_compressed.jpg"
    shutil.move(str(best_path), str(final_path))

    # Clean up other temp files
    for f in temp_files:
        if f != best_path and f.exists():
            f.unlink()

    if print_debug:
        print(f"{'='*70}")
        print(f"✅ COMPRESSION COMPLETE")
        print(f"{'='*70}")
    print(f"Original: {original_size_mb:.1f} MB")
    print(f"Compressed: {best_size:.1f} MB at quality {best_quality}")
    print(f"Saved to: {final_path}")
    if print_debug:
        print(f"{'='*70}\n")
    return str(final_path)


# Keep old function for compatibility with extra tools
def compress_to_target_size(
    input_path, output_dir, target_size_mb=50, format="JPEG", quality=95
):
    """Legacy function - use compress_image() instead."""
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    img = Image.open(input_path)
    original_size = img.size

    # Convert RGBA to RGB for JPEG
    if img.mode in ("RGBA", "LA", "P"):
        background = Image.new("RGB", img.size, (255, 255, 255))
        if img.mode == "P":
            img = img.convert("RGBA")
        background.paste(
            img, mask=img.split()[-1] if img.mode in ("RGBA", "LA") else None
        )
        img = background

    temp_files = []

    if format.upper() == "JPEG":
        ext = "jpg"
        min_quality = 30
        max_quality = quality
        best_attempt = None
        iterations = 0
        max_iterations = 8

        while min_quality <= max_quality and iterations < max_iterations:
            current_quality = (min_quality + max_quality) // 2
            output_path = output_dir / f"{input_path.stem}_q{current_quality}.{ext}"

            img.save(output_path, format, quality=current_quality, optimize=True)
            size_mb = get_file_size_mb(output_path)
            temp_files.append(output_path)

            current_attempt = {
                "quality": current_quality,
                "size_mb": size_mb,
                "path": output_path,
                "resized": False,
            }

            if best_attempt is None or (
                size_mb <= target_size_mb and current_quality > best_attempt["quality"]
            ):
                best_attempt = current_attempt

            if size_mb > target_size_mb:
                max_quality = current_quality - 1
            else:
                min_quality = current_quality + 1

            iterations += 1

        if best_attempt is None:
            best_attempt = current_attempt

    if best_attempt["size_mb"] > target_size_mb:
        # Calculate resize factor needed
        current_pixels = img.size[0] * img.size[1]
        target_pixels = int(
            current_pixels * (target_size_mb / best_attempt["size_mb"]) * 0.9
        )  # 0.9 safety margin
        scale_factor = (target_pixels / current_pixels) ** 0.5

        new_width = int(img.size[0] * scale_factor)
        new_height = int(img.size[1] * scale_factor)

        resized_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
        output_path = (
            output_dir / f"{input_path.stem}_resized_{new_width}x{new_height}.{ext}"
        )
        resized_img.save(output_path, format, quality=quality, optimize=True)

        size_mb = get_file_size_mb(output_path)
        best_attempt = {
            "quality": quality,
            "size_mb": size_mb,
            "path": output_path,
            "resized": True,
            "new_dimensions": (new_width, new_height),
        }

    original_size_mb = get_file_size_mb(input_path)

    # Create final output file with clean name (without quality suffix)
    final_output_path = output_dir / f"{input_path.stem}_compressed.{ext}"

    # Copy best attempt to final output
    import shutil

    shutil.copy2(best_attempt["path"], final_output_path)

    # Delete all temporary test files
    for temp_file in temp_files:
        try:
            if temp_file.exists():
                temp_file.unlink()
        except:
            pass

    return {
        "method": f"{format} (target: {target_size_mb} MB)",
        "input_path": str(input_path),
        "output_path": str(final_output_path),
        "original_size_mb": original_size_mb,
        "compressed_size_mb": best_attempt["size_mb"],
        "compression_ratio": original_size_mb / best_attempt["size_mb"],
        "size_reduction_percent": (
            (original_size_mb - best_attempt["size_mb"]) / original_size_mb
        )
        * 100,
        "quality": best_attempt.get("quality"),
        "resized": best_attempt.get("resized", False),
        "new_dimensions": best_attempt.get("new_dimensions", original_size),
        "original_dimensions": original_size,
    }


def compress_to_png(input_path, output_dir, compression_level=9):
    """
    Compress image to PNG format (lossless).

    Args:
        input_path: Path to input image
        output_dir: Directory to save compressed image
        compression_level: PNG compression level (0-9, 9=maximum compression)

    Returns:
        dict with compression results
    """
    input_path = Path(input_path)
    output_path = output_dir / f"{input_path.stem}_compressed.png"

    # Open and save with PNG compression
    img = Image.open(input_path)

    # Convert RGBA to RGB if necessary (some formats don't support transparency)
    if img.mode == "RGBA":
        # Create white background
        background = Image.new("RGB", img.size, (255, 255, 255))
        background.paste(img, mask=img.split()[3])  # Use alpha channel as mask
        img = background

    # Save with specified compression
    img.save(output_path, "PNG", compress_level=compression_level, optimize=True)

    # Get sizes
    original_size = get_file_size_mb(input_path)
    compressed_size = get_file_size_mb(output_path)
    compression_ratio = original_size / compressed_size if compressed_size > 0 else 0
    size_reduction_percent = ((original_size - compressed_size) / original_size) * 100

    return {
        "method": "PNG",
        "input_path": str(input_path),
        "output_path": str(output_path),
        "original_size_mb": original_size,
        "compressed_size_mb": compressed_size,
        "compression_ratio": compression_ratio,
        "size_reduction_percent": size_reduction_percent,
        "compression_level": compression_level,
    }


def compress_to_tiff_lzw(input_path, output_dir):
    """
    Compress image to TIFF format with LZW compression (lossless).

    Args:
        input_path: Path to input image
        output_dir: Directory to save compressed image

    Returns:
        dict with compression results
    """
    input_path = Path(input_path)
    output_path = output_dir / f"{input_path.stem}_compressed.tiff"

    # Open and save with TIFF LZW compression
    img = Image.open(input_path)

    # Convert RGBA to RGB if necessary
    if img.mode == "RGBA":
        background = Image.new("RGB", img.size, (255, 255, 255))
        background.paste(img, mask=img.split()[3])
        img = background

    # Save with LZW compression
    img.save(output_path, "TIFF", compression="tiff_lzw")

    # Get sizes
    original_size = get_file_size_mb(input_path)
    compressed_size = get_file_size_mb(output_path)
    compression_ratio = original_size / compressed_size if compressed_size > 0 else 0
    size_reduction_percent = ((original_size - compressed_size) / original_size) * 100

    return {
        "method": "TIFF (LZW)",
        "input_path": str(input_path),
        "output_path": str(output_path),
        "original_size_mb": original_size,
        "compressed_size_mb": compressed_size,
        "compression_ratio": compression_ratio,
        "size_reduction_percent": size_reduction_percent,
    }


def compress_to_zip(input_path, output_dir, compression_level=9):
    """
    Compress image file using ZIP (lossless file compression).
    Preserves original format.

    Args:
        input_path: Path to input image
        output_dir: Directory to save compressed archive
        compression_level: ZIP compression level (0-9)

    Returns:
        dict with compression results
    """
    input_path = Path(input_path)
    output_path = output_dir / f"{input_path.stem}.zip"

    # Create ZIP archive with maximum compression
    with zipfile.ZipFile(
        output_path, "w", zipfile.ZIP_DEFLATED, compresslevel=compression_level
    ) as zipf:
        zipf.write(input_path, input_path.name)

    # Get sizes
    original_size = get_file_size_mb(input_path)
    compressed_size = get_file_size_mb(output_path)
    compression_ratio = original_size / compressed_size if compressed_size > 0 else 0
    size_reduction_percent = ((original_size - compressed_size) / original_size) * 100

    return {
        "method": "ZIP",
        "input_path": str(input_path),
        "output_path": str(output_path),
        "original_size_mb": original_size,
        "compressed_size_mb": compressed_size,
        "compression_ratio": compression_ratio,
        "size_reduction_percent": size_reduction_percent,
    }


def compress_image_all_methods(
    input_path, output_dir="compressed_images", show_results=True
):
    """
    Compress image using all available lossless methods and compare results.

    Args:
        input_path: Path to input image
        output_dir: Directory to save compressed images
        show_results: Whether to print results table

    Returns:
        list of dicts with compression results for each method
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    results = []

    print(f"\n{'='*70}")
    print(f"🗜️  COMPRESSING IMAGE (LOSSLESS)")
    print(f"{'='*70}")
    print(f"Input: {input_path}")
    print(f"Output directory: {output_dir.absolute()}")
    print(f"{'='*70}\n")

    # Check image info for very large images
    try:
        info = check_image_info(input_path)
        print()
    except Exception as e:
        print(f"⚠️  Could not read image info: {e}\n")

    # Try PNG compression
    try:
        print("📦 Compressing to PNG...")
        result = compress_to_png(input_path, output_dir, compression_level=9)
        results.append(result)
        print(f"   ✅ Done: {result['compressed_size_mb']:.2f} MB")
    except Exception as e:
        print(f"   ❌ Failed: {e}")

    # Try TIFF LZW compression
    try:
        print("📦 Compressing to TIFF (LZW)...")
        result = compress_to_tiff_lzw(input_path, output_dir)
        results.append(result)
        print(f"   ✅ Done: {result['compressed_size_mb']:.2f} MB")
    except Exception as e:
        print(f"   ❌ Failed: {e}")

    # Try ZIP compression
    try:
        print("📦 Compressing to ZIP...")
        result = compress_to_zip(input_path, output_dir, compression_level=9)
        results.append(result)
        print(f"   ✅ Done: {result['compressed_size_mb']:.2f} MB")
    except Exception as e:
        print(f"   ❌ Failed: {e}")

    if show_results and results:
        print(f"\n{'='*70}")
        print("📊 COMPRESSION RESULTS")
        print(f"{'='*70}")
        print(
            f"{'Method':<15} {'Original (MB)':<15} {'Compressed (MB)':<18} {'Reduction':<12} {'Ratio':<8}"
        )
        print(f"{'-'*70}")

        for r in results:
            print(
                f"{r['method']:<15} "
                f"{r['original_size_mb']:<15.2f} "
                f"{r['compressed_size_mb']:<18.2f} "
                f"{r['size_reduction_percent']:<11.1f}% "
                f"{r['compression_ratio']:<8.2f}x"
            )

        # Find best compression
        best = min(results, key=lambda x: x["compressed_size_mb"])
        print(f"\n🏆 Best compression: {best['method']}")
        print(f"   Output: {best['output_path']}")
        print(f"{'='*70}\n")

    return results


print("✅ Compression functions defined!")

✅ Compression functions defined!


---

## 🎯 COMPRESS YOUR IMAGE

**Simple**: Just set path and target size below!

In [8]:
# ============================================================================
# 📝 SET YOUR IMAGE PATH AND TARGET SIZE
# ============================================================================
image_path = "C:\\Users\\Erik Liu\\Downloads\\028_all.png"
target_size = 50  # MB

# ============================================================================
# RUN COMPRESSION (with debug output)
# ============================================================================
# For Overleaf: use max_dimension=16384 (Overleaf's limit is 16384 pt)
compress_image(image_path, target_size, print_debug=True, max_dimension=16000)


🔍 DEBUG MODE - Compression Process
Input: 028_all.png
Original size: 69.51 MB
Target size: 50.00 MB
Max dimension limit: 16000 px

📐 Checking dimension limits...
   Current dimensions: 16384 x 9208 px
   Max dimension limit: 16000 px
   ❌ EXCEEDS LIMIT by 384 px

📐 Step 0: Resizing to fit dimension limit...
   Resizing: 16384 x 9208 → 16000 x 8992 px
   Scale factor: 0.9766

📦 Step 1: Trying PNG lossless compression...
📦 Step 1: Trying PNG lossless compression...
   PNG size: 57.99 MB
   ❌ PNG too large (> 50 MB)
   → Moving to JPEG compression...

🔍 Step 2: Binary search for optimal JPEG quality
   Search range: quality 50-99
   Converge when gap < 2

   PNG size: 57.99 MB
   ❌ PNG too large (> 50 MB)
   → Moving to JPEG compression...

🔍 Step 2: Binary search for optimal JPEG quality
   Search range: quality 50-99
   Converge when gap < 2

   Iteration 1: Quality 74 → 6.27 MB ✅ FITS
      → File fits! Trying higher quality (range now 74-99)
   Iteration 1: Quality 74 → 6.27 MB ✅ FIT

'compressed_images\\028_all_compressed.jpg'

---

## 🛠️ EXTRA TOOLS (Optional)

Below are additional tools for advanced usage. You don't need to run these cells unless you want to:
- Test different compression methods
- Compare PNG compression levels  
- Do visual quality checks
- Batch process multiple images

### Test All Lossless Methods

Compare PNG, TIFF (LZW), and ZIP compression on your image.

In [4]:
# Uncomment and set path to test all lossless compression methods
# image_path = "C:\\Users\\Erik Liu\\Downloads\\028_all.png"
# results = compress_image_all_methods(image_path, output_dir, show_results=True)

### Compare PNG Compression Levels

Test all PNG compression levels (0-9) to see the difference.

In [5]:
def test_png_compression_levels(input_path, output_dir="compressed_images"):
    """
    Test different PNG compression levels and compare results.
    """
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    if not input_path.exists():
        print(f"❌ Image not found: {input_path}")
        return

    print(f"\n{'='*70}")
    print(f"🔬 TESTING PNG COMPRESSION LEVELS")
    print(f"{'='*70}")
    print(f"Input: {input_path}")
    print(f"{'='*70}\n")

    results = []

    for level in range(0, 10):
        output_path = output_dir / f"{input_path.stem}_level_{level}.png"

        # Compress with this level
        img = Image.open(input_path)
        if img.mode == "RGBA":
            background = Image.new("RGB", img.size, (255, 255, 255))
            background.paste(img, mask=img.split()[3])
            img = background

        img.save(output_path, "PNG", compress_level=level, optimize=True)

        # Get size
        compressed_size = get_file_size_mb(output_path)

        results.append(
            {"level": level, "size_mb": compressed_size, "path": output_path}
        )

        print(f"Level {level}: {compressed_size:.3f} MB")

    print(f"\n{'='*70}")
    print("📊 SUMMARY")
    print(f"{'='*70}")

    original_size = get_file_size_mb(input_path)
    best = min(results, key=lambda x: x["size_mb"])
    worst = max(results, key=lambda x: x["size_mb"])

    print(f"Original size: {original_size:.3f} MB")
    print(
        f"Best (level {best['level']}): {best['size_mb']:.3f} MB "
        f"({((original_size - best['size_mb']) / original_size * 100):.1f}% reduction)"
    )
    print(
        f"Worst (level {worst['level']}): {worst['size_mb']:.3f} MB "
        f"({((original_size - worst['size_mb']) / original_size * 100):.1f}% reduction)"
    )
    print(
        f"Difference: {(worst['size_mb'] - best['size_mb']):.3f} MB "
        f"({((worst['size_mb'] - best['size_mb']) / best['size_mb'] * 100):.1f}% larger)"
    )

    print(f"\n💡 Recommendation: Use level 9 for maximum compression")
    print(f"{'='*70}\n")

    return results


# Uncomment and set path to test PNG compression levels
# test_png_compression_levels("C:\\Users\\Erik Liu\\Downloads\\028_all.png")

### Visual Quality Comparison

Display original and compressed images side-by-side.

In [6]:
def visual_comparison(original_path, compressed_path):
    """
    Display original and compressed images side-by-side.
    For lossless compression, they should look identical.
    """
    original_path = Path(original_path)
    compressed_path = Path(compressed_path)

    if not original_path.exists():
        print(f"❌ Original image not found: {original_path}")
        return

    if not compressed_path.exists():
        print(f"❌ Compressed image not found: {compressed_path}")
        return

    # Load images
    original = Image.open(original_path)
    compressed = Image.open(compressed_path)

    # Convert to arrays for comparison
    original_array = np.array(original)
    compressed_array = np.array(compressed)

    # Check if identical
    if original_array.shape == compressed_array.shape:
        identical = np.array_equal(original_array, compressed_array)
    else:
        identical = False

    # Display
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(original)
    axes[0].set_title(f"Original\n{get_file_size_mb(original_path):.2f} MB")
    axes[0].axis("off")

    axes[1].imshow(compressed)
    axes[1].set_title(f"Compressed\n{get_file_size_mb(compressed_path):.2f} MB")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"\n{'='*70}")
    print("🔍 QUALITY CHECK")
    print(f"{'='*70}")
    print(f"Original shape: {original_array.shape}")
    print(f"Compressed shape: {compressed_array.shape}")
    print(f"Pixel-perfect match: {'✅ YES' if identical else '❌ NO'}")

    if identical:
        print("\n✅ Compression is LOSSLESS - all pixels are identical!")
    else:
        print("\n⚠️  Images differ - check if this is expected")
        print(
            "   (Different formats may change color space or add compression artifacts)"
        )

    print(f"{'='*70}\n")


# Uncomment and set paths to compare original vs compressed
# visual_comparison(
#     "C:\\Users\\Erik Liu\\Downloads\\028_all.png",
#     "compressed_images/028_all_compressed.jpg"
# )

### Batch Process Multiple Images

Compress multiple images at once.

In [7]:
def compress_batch(image_paths, output_ir="compressed_images", method="PNG"):
    """
    Compress multiple images at once.

    Args:
        image_paths: List of image paths to compress
        output_dir: Output directory for compressed images
        method: Compression method ('PNG', 'TIFF', 'ZIP', or 'ALL')

    Returns:
        list of compression results
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    all_results = []

    print(f"\n{'='*70}")
    print(f"📦 BATCH COMPRESSION")
    print(f"{'='*70}")
    print(f"Images to process: {len(image_paths)}")
    print(f"Method: {method}")
    print(f"Output: {output_dir.absolute()}")
    print(f"{'='*70}\n")

    for i, img_path in enumerate(image_paths, 1):
        img_path = Path(img_path)

        if not img_path.exists():
            print(f"⚠️  Skipping {img_path.name} - file not found")
            continue

        print(f"[{i}/{len(image_paths)}] Processing: {img_path.name}")

        try:
            if method.upper() == "ALL":
                results = compress_image_all_methods(
                    img_path, output_dir, show_results=False
                )
                all_results.extend(results)
            elif method.upper() == "PNG":
                result = compress_to_png(img_path, output_dir)
                all_results.append(result)
                print(
                    f"   ✅ {result['compressed_size_mb']:.2f} MB "
                    f"({result['size_reduction_percent']:.1f}% reduction)"
                )
            elif method.upper() == "TIFF":
                result = compress_to_tiff_lzw(img_path, output_dir)
                all_results.append(result)
                print(
                    f"   ✅ {result['compressed_size_mb']:.2f} MB "
                    f"({result['size_reduction_percent']:.1f}% reduction)"
                )
            elif method.upper() == "ZIP":
                result = compress_to_zip(img_path, output_dir)
                all_results.append(result)
                print(
                    f"   ✅ {result['compressed_size_mb']:.2f} MB "
                    f"({result['size_reduction_percent']:.1f}% reduction)"
                )
        except Exception as e:
            print(f"   ❌ Failed: {e}")

    # Summary
    if all_results:
        total_original = sum(r["original_size_mb"] for r in all_results)
        total_compressed = sum(r["compressed_size_mb"] for r in all_results)
        total_reduction = (total_original - total_compressed) / total_original * 100

        print(f"\n{'='*70}")
        print("📊 BATCH SUMMARY")
        print(f"{'='*70}")
        print(f"Images processed: {len(all_results)}")
        print(f"Total original size: {total_original:.2f} MB")
        print(f"Total compressed size: {total_compressed:.2f} MB")
        print(
            f"Total space saved: {total_original - total_compressed:.2f} MB ({total_reduction:.1f}%)"
        )
        print(f"{'='*70}\n")

    return all_results


# Uncomment and add your image paths to batch compress
# image_list = [
#     "C:\\path\\to\\image1.png",zzzz
#     "C:\\path\\to\\image2.jpg",
#     "C:\\path\\to\\image3.png",
# ]
# batch_results = compress_batch(image_list, method="PNG")

---

## 📚 Summary

**Algorithm**:
1. Try PNG lossless → if fits, done!
2. Binary search JPEG quality 50-99 (logarithmic, ~5 iterations)
3. Find highest quality under target size
4. Stop at quality < 50 (impossible target)

**Why it's optimal**:
- ✅ **Fastest**: O(log n) binary search
- ✅ **Best quality**: Always finds maximum quality possible
- ✅ **Converges**: Stops when improvement < 0.1 MB
- ✅ **Simple**: One function call

**Example** (150M pixel image):
- Original: 85 MB
- Target: 50 MB  
- Iterations: Q75→Q62→Q68→Q65→Q66
- Result: 49.8 MB at quality 65 ✨

**Output**: `compressed_images/{name}_compressed.jpg`